# 03 — Complex<T> e DFT
Kernel **JupyterNet** (C#) su runtime **.NET 11**. Prerequisito: `dotnet build src/MathemaSharp.slnx -c Release`. Da terminale: `scripts/run-notebooks.ps1` (imposta il roll-forward al runtime 11); in VS Code serve `DOTNET_ROLL_FORWARD=LatestMajor` e, finché .NET 11 è RC, `DOTNET_ROLL_FORWARD_TO_PRERELEASE=1`.

`Complex<T>` (.NET 11) è generico su `T : IFloatingPointIeee754<T>, IMinMaxValue<T>`: la libreria lo usa per `Signals.Dft<T>` e per tutta la parte quantistica. Non esiste un ordinamento naturale. Angoli in radianti. DFT forward non normalizzata O(N²), non una FFT.

In [ ]:
#load "../shared/MathemaSharp.csx"
using System.Numerics;
using MathemaSharp.Core;
var z = new Complex<double>(3,4);
Console.WriteLine($"{z}: modulo={z.GetMagnitude()}, fase={z.GetPhase()}");
Console.WriteLine(Complex<double>.Sqrt(-Complex<double>.One));
Console.WriteLine(Signals.HermitianDot<double>(new[] {z},new[] {z}));
Console.WriteLine(NumericAlgorithms.Dot<Complex<double>>(new[] {z},new[] {z}));

In [ ]:
int n = 16;
var samples = Enumerable.Range(0,n).Select(i => new Complex<double>(Math.Cos(2*Math.PI*3*i/n),0)).ToArray();
var spectrum = Signals.Dft<double>(samples);
Chart.Stems("|X[k]| — coseno a frequenza 3 su 16 campioni", spectrum.Select(c => c.GetMagnitude()).ToArray(), firstLag: 0);
var leaking = Signals.Dft<double>(Enumerable.Range(0,n).Select(i => new Complex<double>(Math.Cos(2*Math.PI*3.5*i/n),0)).ToArray());
Chart.Stems("|X[k]| — frequenza 3.5: leakage spettrale", leaking.Select(c => c.GetMagnitude()).ToArray(), firstLag: 0);

## Risultato atteso
Picchi a k=3 e k=13, ampiezza 8. Con frequenza 3.5 l'energia si sparpaglia su tutti i bin: leakage spettrale. Prodotto hermitiano con coniugazione a sinistra: ⟨z,z⟩=25, prodotto bilineare z·z=−7+24i. La stessa `Signals.Dft<T>` funziona con float e con Decimal32/64/128: vedere notebook 05 e `--topic formats` della console.